In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01. Ingest Structured CSV

# COMMAND ----------

import re
from pyspark.sql.functions import col, concat_ws, lit

# --- Configuration ---
CATALOG_NAME = "bricksiitm"
SCHEMA_NAME = "ayurgenix"
VOLUME_NAME = "files"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
CSV_PATH = f"{VOLUME_PATH}/AyurGenixAI_Dataset.csv"

DELTA_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.ayurgenix_structured"

# COMMAND ----------

# Set catalog + schema
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

# Debug: check files
display(dbutils.fs.ls(VOLUME_PATH))

# COMMAND ----------

print("📊 Loading CSV...")

csv_df = spark.read \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(CSV_PATH)

print("Columns:", csv_df.columns)

# COMMAND ----------

# Clean column names
def clean_column(name):
    name = name.strip()
    name = re.sub(r"[ &()/]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.lower()

csv_df = csv_df.toDF(*[clean_column(c) for c in csv_df.columns])
csv_df = csv_df.fillna("Not Available")

# COMMAND ----------

# Create text chunk safely
structured_df = csv_df.withColumn(
    "text_chunk",
    concat_ws(" | ",
        lit("Disease"), col("disease"),
        lit("Symptoms"), col("symptoms"),
        lit("Doshas"), col("doshas"),
        lit("Herbs"), col("ayurvedic_herbs"),
        lit("Recommendations"), col("patient_recommendations")
    )
).select(
    lit("CSV").alias("source"),
    col("disease").alias("topic"),
    col("text_chunk")
)

# COMMAND ----------

print("💾 Saving to Delta table...")

structured_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DELTA_TABLE)

display(spark.table(DELTA_TABLE).limit(5))